# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Access and print dataset metadata attributes
md = dataset.metadata
print(f"{md.name}: {md.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Let's inspect the dataset for available record sets and their fields. For each entity, we reference the unique `@id` field.

In [ ]:
# List all available record sets in the dataset
record_sets = list(dataset.record_sets.keys())
print('Available record set IDs:')
for rs_id in record_sets:
    print(f'  - {rs_id}')

# For each record set, list fields and columns
for rs_id in record_sets:
    print(f'\nFields in RecordSet {rs_id}:')
    record_set_obj = dataset.record_sets[rs_id]
    fields = [f['@id'] for f in record_set_obj['fields']]
    print(fields)
    if 'columns' in record_set_obj:
        print('Columns:')
        cols = [c['@id'] for c in record_set_obj['columns']]
        print(cols)
    else:
        print('No columns listed.')

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

Here, we'll load all record sets into pandas DataFrames, referencing entities by their `@id`.

In [ ]:
# Extract data from each record set
dataframes = {}
for rs_id in record_sets:
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df

# Display columns for main record set
main_rs_id = record_sets[0]  # Usually the main survey record set
print(f'Columns in main record set ({main_rs_id}):')
print(dataframes[main_rs_id].columns.tolist())

dataframes[main_rs_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

We will reference all fields and columns by their `@id`.

In [ ]:
# Identify numeric fields for analysis; example field IDs for GAD-7, PHQ-9, PSQ (update as needed)
numeric_field = 'https://api.app.sen.science/frontiers/7160411/gad7_score'  # Example field @id for GAD-7 score
record_set_id = main_rs_id

# Set filter threshold for analysis (e.g., people with GAD-7 score over 10)
threshold = 10
if numeric_field in dataframes[record_set_id].columns:
    filtered_df = dataframes[record_set_id][dataframes[record_set_id][numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold}:")
    print(filtered_df.head())

    # Normalize numeric field
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Grouping by a demographic field, e.g. village (update to a real @id)
    group_field = 'https://api.app.sen.science/frontiers/7160411/village'  # Example field @id for village
    if group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
        print(f"Grouped data by {group_field}:")
        print(grouped_df.head())
else:
    print(f"Numeric field {numeric_field} not found in DataFrame columns. Available columns:")
    print(dataframes[record_set_id].columns.tolist())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Below we plot the distribution of GAD-7 scores and the average GAD-7 score by village, referencing fields by their `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot distribution of GAD-7 scores
if numeric_field in dataframes[record_set_id].columns:
    plt.figure(figsize=(8,6))
    sns.histplot(dataframes[record_set_id][numeric_field].dropna(), bins=20, kde=True)
    plt.title('Distribution of GAD-7 Scores')
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    # Plot mean GAD-7 score by village (group_field)
    if group_field in dataframes[record_set_id].columns:
        means_by_village = dataframes[record_set_id].groupby(group_field)[numeric_field].mean()
        means_by_village = means_by_village.dropna()
        means_by_village.plot(kind='bar', figsize=(10,5))
        plt.title(f'Average GAD-7 Score by Village ({group_field})')
        plt.xlabel('Village')
        plt.ylabel('Mean GAD-7 Score')
        plt.xticks(rotation=45)
        plt.show()
else:
    print(f"Numeric field {numeric_field} not found. Visualization skipped.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The dataset provides detailed mental health survey data from Kilifi County, Kenya, including personal demographics and psychological assessment scores.
- All entities and fields were referenced by their unique `@id` for precise mapping.
- Exploratory analysis and filtering identified participants with elevated GAD-7 scores, and revealed trends when grouped by village.
- This structured approach ensures data is AI-ready and accessible for further public health analysis or research.